# Prepare Environment

#### Suppress Bash Warnings

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import os

# Remove the exported function definitions that cause the noise
for key in list(os.environ.keys()):
    if key.startswith("BASH_FUNC_"):
        del os.environ[key]

if os.path.exists('.bedrock_agentcore.yaml'):
    os.remove('.bedrock_agentcore.yaml')

#### Create Demo Role

In [ ]:
%run 09_create_demo_role.ipynb

# Standalone Agent on Local Environment (Jupyter Notebook)
<img src="images/local_agent.png" width="1100">

In [ ]:
import json
import argparse

from strands import Agent
from strands.models import BedrockModel

model_id = "global.anthropic.claude-opus-4-6-v1"
model = BedrockModel(
    model_id=model_id,
)

agent = Agent(
    model=model,
    system_prompt="You're a helpful assistant."
)

In [ ]:
response = agent("Top 5 biggest countries by size ?")

In [ ]:
response = agent("What is the weather in Athens now ?")

#### Adding a tool

In [ ]:
from strands_tools.tavily import tavily_search

os.environ["TAVILY_API_KEY"] = "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"
#TODO : Add DUCKDUODO scenario

agent = Agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You're a helpful assistant."
)

In [ ]:
response = agent("What is the weather in Athens now ?")

#### Adding a custom tool

In [ ]:
from strands import tool

# Weather tool
@tool
def weather(city: str):
    """Get weather information.

    Args:
        city: City for which weather will be returned

    Returns:
        Weather of provided city as a string.
    """
    print(city)
    if city.lower() == "athens":
        weather = "very sunny"
    else:
        weather = "sunny"
    return weather

In [ ]:
agent = Agent(
    model=model,
    tools=[weather, tavily_search],
    system_prompt="You're a helpful assistant."
)

response = agent("What is the weather in Athens now ?")

In [ ]:
response = agent("What is the weather in Athens now ? Cross check with information from the web")

# Agent on AgentCore
<img src="images/standalone_agent_on_agent_core.png" width="1100">

In [ ]:
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime


boto_session = Session()
region = boto_session.region_name

<img src="images/agent_core_configure_and_launch.png" width="1100">

In [ ]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="01_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

In [ ]:
launch_result = agent_runtime.launch(auto_update_on_conflict=True)

In [ ]:
invoke_response = agent_runtime.invoke({"prompt": "How is the weather now in Athens ? Cross check with information from the web."})
invoke_response

# MCP Server on  AgentCore
<img src="images/mcp_on_agent_core.png" width="800">

In [ ]:
mcp_server_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = mcp_server_runtime.configure(
    entrypoint="01_mcp_server.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name='mpc_server',
)

In [ ]:
mcp_server_runtime.launch(auto_update_on_conflict=True)

# Agent to MCP Server on  AgentCore
<img src="images/agent_to_mpc_on_agent_core.png" width="1100">

In [ ]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="02_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

In [ ]:
agent_runtime.launch(auto_update_on_conflict=True)

In [ ]:
response = agent_runtime.invoke({"prompt": "How is the weather now in Athens ? Cross check your answer with web information."})
response

# AgentCore Gateway
<img src="images/gateway.png" width="1300">

#### Create Gateway

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control")

# Create new gateway
response = agentcore_control_client.create_gateway(
    name='gateway',
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="AWS_IAM",
)

gateway_id = response["gatewayId"]
gateway_url = response["gatewayUrl"]

print(f"Gateway URL: {gateway_url}")

#### Create gateway MCP target

In [ ]:
MCP_SERVER_ARN = "arn:aws:bedrock-agentcore:us-east-1:869795389844:runtime/mpc_server-IbPAU23Iwf"
encoded_arn = MCP_SERVER_ARN.replace(":", "%3A").replace("/", "%2F")

mcp_endpoint = (
    f"https://bedrock-agentcore.{region}.amazonaws.com"
    f"/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
)

target = agentcore_control_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name='mcp-server',
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_endpoint,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "GATEWAY_IAM_ROLE",
            "credentialProvider": {
                "iamCredentialProvider": {
                    "service": "bedrock-agentcore",
                    "region": region,
                }
            },
        }
    ],
)

print(f"Target ID: {target['targetId']}")
print(f"Target status: {target['status']}")

#### Create gateway REST API target

In [ ]:
ACCOUNT_ID = "869795389844"
S3_OPENAPI_URI = f"s3://agentcore-rest-api-{ACCOUNT_ID}/openapi.json"
REST_API_KEY_PROVIDER_ARN = "arn:aws:bedrock-agentcore:us-east-1:869795389844:token-vault/default/apikeycredentialprovider/rest-api-key"

print(f"\nAdding REST API target...")
rest_target = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="rest-api",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "s3": {
                    "uri": S3_OPENAPI_URI,
                    "bucketOwnerAccountId": ACCOUNT_ID,
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": REST_API_KEY_PROVIDER_ARN,
                    "credentialLocation": "HEADER",
                    "credentialParameterName": "x-api-key",
                }
            },
        }
    ],
)

print(f"REST API Target ID: {rest_target['targetId']}")
print(f"REST API Target status: {rest_target['status']}")

# Agent with AgentCore Gateway
<img src="images/agent_to_gateway.png" width="1200">

In [ ]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="03_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

In [ ]:
launch_result = agent_runtime.launch(auto_update_on_conflict=True)

In [ ]:
import uuid

agent_core_client = boto3.client("bedrock-agentcore")
agent_arn = launch_result.agent_arn  # get ARN from the toolkit

session_id = str(uuid.uuid4())  # one session for the whole conversation

def chat(prompt, account_id="cust-003"):
    payload = json.dumps({"prompt": prompt, "accountId": account_id}).encode()
    response = agent_core_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=session_id,
        payload=payload,
        qualifier="DEFAULT",
    )
    content = []
    for chunk in response.get("response", []):
        content.append(chunk.decode("utf-8"))
    result = json.loads("".join(content))
    print(result)

In [ ]:
chat("what bets have I played so far ?")

In [ ]:
chat("Find me the best value football bet right now — check the events, compare the odds, pick the one with the highest potential return, and present it to me.")

In [ ]:
chat("great! place it for 55$")

# Observability
<img src="images/observability.png">

# AgentCore Memory
<img src="images/memory.png" width="1300">

# Agent with Memory
<img src="images/agent_with_memory.png" width="1000">

In [ ]:
agent_runtime = Runtime()

agent_runtime.configure(
    agent_name="agent",
    entrypoint="04_agent.py",
    auto_create_execution_role=False,
    execution_role="AgentCoreDemoRole",
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region
)

In [ ]:
launch_result = agent_runtime.launch(auto_update_on_conflict=True)

In [24]:
chat("what bets have I played so far ?")

Everything is still the same — your **5 active bets** are all on **Dortmund to win** against Bayern Munich:

| # | Bet ID | Stake | Odds | Potential Return | Status | Placed |
|---|--------|-------|------|------------------|--------|--------|
| 1 | bet-b6016c7f | $50.00 | 4.5 | $225.00 | 🟢 Active | Apr 23 |
| 2 | bet-738d5ae9 | $51.00 | 4.5 | $229.50 | 🟢 Active | Apr 23 |
| 3 | bet-9a6c6e82 | $99.00 | 4.5 | $445.50 | 🟢 Active | Apr 23 |
| 4 | bet-bb9e4691 | $52.00 | 4.5 | $234.00 | 🟢 Active | Apr 24 |
| 5 | bet-c446167b | $55.00 | 4.5 | $247.50 | 🟢 Active | Apr 24 |

**📊 Summary:**
- **Total Bets:** 5
- **Total Staked:** $307.00
- **Total Potential Return:** $1,381.50

No changes since last time — all bets are still active and waiting for the **Bayern Munich vs Dortmund** match on **Apr 26 at 5:30 PM UTC**. Fingers crossed for Dortmund! 🤞💛

Anything else I can help with?


In [25]:
chat("is there any bet available for me to play ? show me only for my favorite games")

Same events are still available for your favorite sports! 🎯

### ⚽ Football
| # | Event | League | Kickoff | Status |
|---|-------|--------|---------|--------|
| 1 | **Arsenal** vs Chelsea | **Premier League** ⭐ | Apr 25, 3:00 PM UTC | 🔜 Upcoming |
| 2 | Barcelona vs Real Madrid | La Liga | Apr 25, 8:00 PM UTC | 🔜 Upcoming |
| 3 | Bayern Munich vs Dortmund | Bundesliga | Apr 26, 5:30 PM UTC | 🔜 Upcoming |

### 🏀 Basketball
| # | Event | League | Kickoff | Status |
|---|-------|--------|---------|--------|
| 1 | Lakers vs Celtics | NBA | Apr 25, 7:30 PM UTC | 🔴 LIVE |
| 2 | Warriors vs Bucks | NBA | Apr 26, 9:00 PM UTC | 🔜 Upcoming |

🔥 **Quick reminders:**
- You already have **5 bets ($307 staked)** on **Dortmund** in the Bayern match
- **Arsenal vs Chelsea** is still open — a great chance to back your Gunners! 🔴
- **Lakers vs Celtics** is still **LIVE** if you want some basketball action 🏀

Want me to pull up the odds for any of these? I know you like **match result bets** with stakes